# Behavioural Figures

Clean workflow for the current two-animal Bonsai dataset. Re-run from the top whenever another day has been copied into `data_path`; the loader discovers valid sessions from the directory tree.

Figures generated below:

1. Success rate per session, animal, and day
2. Miss rate and wrong-poke error rate on the same axes
3. Signed and unsigned error angle per animal and day
4. Correct and incorrect premature poke rate


## 1. Setup

Only edit the configuration values in this cell unless the experiment event vocabulary changes. The parser definitions live here so the rest of the notebook stays clean.

In [ ]:
from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from notebook_setup import REPO_ROOT, SRC_ROOT
except ModuleNotFoundError:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "Q_C_Analysis_Workflow").is_dir():
        REPO_ROOT = REPO_ROOT.parent
    SRC_ROOT = REPO_ROOT / "src"

for path in (REPO_ROOT, SRC_ROOT):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from Q_C_Analysis_Workflow.utils import (
    build_master_trial_table,
    build_trial_table_for_directory,
    collect_session_settings_directory,
    count_events_per_session,
    count_premature_pokes_per_directory,
    display_table,
    parse_poke_outcome,
)
from data_conduit.datasources.monosource import ExperimentEvents
from Q_C_Analysis_Workflow.utils.trial_table import _events_to_long_frame

warnings.filterwarnings("ignore", category=FutureWarning)

# Current dataset root. Add the second animal under this directory; no other code needs to change.
data_path = '/home/callum/Keshavarzi-Lab-Workspace/data-conduit/Q_C_Analysis_Workflow/data'
base_dir = Path(data_path)

# Rig geometry used for port-index -> angular-error conversion.
N_PORTS = 18

# Lab/event definitions.
TRIAL_START_FOR_PARSER = "Start Trial"
FIRST_TRIAL_START_MARKER = re.compile(r"^Start trial logic$", re.IGNORECASE)
SUBSEQUENT_TRIAL_START_DELAY_S = 1.0
TRIAL_END_MARKER = re.compile(r"^Poke:")
POKE_MARKER = TRIAL_END_MARKER

PHASE_MARKERS = {
    "tz_available": "Target zone available",
    "tz_triggered": "Target zone triggered",
    "await_poke": "Await poke",
}
SEGMENTS = {
    "outbound": ("start", "tz_triggered"),
    "inbound": ("tz_triggered", "end"),
}

# Optional output folder. Figures are shown inline; set SAVE_FIGURES=True to also write PNGs.
SAVE_FIGURES = False
figure_dir = REPO_ROOT / "Q_C_Analysis_Workflow" / "design_space" / "figures"

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

## 2. Load SessionSettings and Behavioural Trials

`SessionSettings` supplies the Bonsai trial table. `ExperimentEvents` supplies the behavioural trial boundaries and outcomes.

In [ ]:
collected = collect_session_settings_directory(base_dir=base_dir, verbose=False)
sessions_df = collected["sessions"].copy()
bonsai_trials_df = collected["trials"].copy()

if sessions_df.empty:
    raise ValueError(f"No valid SessionSettings files were found under {base_dir}")


print(f"base_dir       : {base_dir}")
print(f"animals found  : {', '.join(sorted(sessions_df['mouse'].unique()))}")
print(f"sessions       : {len(sessions_df):,}")
print(f"Bonsai trials  : {len(bonsai_trials_df):,}")

display_table(sessions_df[["USID", "mouse", "day", "session", "n_trials"]])

In [ ]:
behavioural_trials = build_trial_table_for_directory(
    base_dir=base_dir,
    sessions_df=sessions_df,
    start=TRIAL_START_FOR_PARSER,
    end=TRIAL_END_MARKER,
    phases=PHASE_MARKERS,
    segments=SEGMENTS,
    outcome_parser=parse_poke_outcome,
)

if behavioural_trials.empty:
    raise ValueError("No behavioural trials were parsed from ExperimentEvents. Check event markers above.")

print(f"behavioural trials parsed: {len(behavioural_trials):,}")
display_table(behavioural_trials.head(20))

## 3. Apply the Lab Trial-Start Rule

The parser uses `Start Trial` to find trial blocks. For analysis timing, this cell applies the lab convention from the notes:

- first behavioural trial starts at `Start trial logic` when present
- subsequent behavioural trials start at the previous trial end plus `SUBSEQUENT_TRIAL_START_DELAY_S`

The original logged parser start is preserved as `logged_start_time`.

In [ ]:
def first_event_time_by_session(base_dir, sessions, marker, event_column="Event"):
    """Return {USID: first matching ExperimentEvents time} for each session."""
    first_times = {}
    for row in sessions.itertuples(index=False):
        session_path = base_dir / row.mouse / row.day / row.session
        try:
            events_obj = ExperimentEvents(experiment_directory_path=str(session_path), verbose=False).df
            events_df = _events_to_long_frame(events_obj, event_column)
        except Exception:
            continue

        if events_df.empty or event_column not in events_df.columns:
            continue

        event_text = events_df[event_column].astype(str)
        mask = event_text.str.contains(marker, regex=True, na=False)
        if mask.any():
            first_times[row.USID] = events_df.index[mask][0]
    return first_times


def apply_lab_trial_start_rule(trials, sessions, base_dir):
    """Update start_time and outbound_duration using the lab's behavioural start convention."""
    adjusted = trials.sort_values(["USID", "trial_index"]).copy()
    adjusted["logged_start_time"] = adjusted["start_time"]

    first_start_times = first_event_time_by_session(
        base_dir=base_dir,
        sessions=sessions,
        marker=FIRST_TRIAL_START_MARKER,
    )

    previous_end = adjusted.groupby("USID")["end_time"].shift(1)
    adjusted["start_time"] = previous_end + SUBSEQUENT_TRIAL_START_DELAY_S

    first_trial = adjusted.groupby("USID").cumcount() == 0
    first_from_events = adjusted.loc[first_trial, "USID"].map(first_start_times)
    adjusted.loc[first_trial, "start_time"] = first_from_events.fillna(
        adjusted.loc[first_trial, "logged_start_time"]
    )

    adjusted["outbound_duration"] = adjusted["tz_triggered_time"] - adjusted["start_time"]
    adjusted["inbound_duration"] = adjusted["end_time"] - adjusted["tz_triggered_time"]
    return adjusted


behavioural_trials = apply_lab_trial_start_rule(
    trials=behavioural_trials,
    sessions=sessions_df,
    base_dir=base_dir,
)

master_trials = build_master_trial_table(
    behavioural_trials=behavioural_trials,
    bonsai_trials=bonsai_trials_df,
)

print(f"master trials: {len(master_trials):,}")
display_table(master_trials.head(20))

## 4. Quality Checks

This compares three counts per session: configured Bonsai trials, parsed behavioural trials, and raw `Poke:` events in `ExperimentEvents`. `parser_matches_events` should usually be true; Bonsai-configured trials can be larger because settings often include planned trials that never fired.

In [ ]:
events_per_session = count_events_per_session(
    base_dir=base_dir,
    sessions_df=sessions_df,
    marker=TRIAL_END_MARKER,
)
parser_per_session = behavioural_trials.groupby("USID").size().rename("n_trials_parser")

trial_count_table = (
    sessions_df[["USID", "mouse", "day", "session", "n_trials"]]
    .merge(parser_per_session.reset_index(), on="USID", how="left")
    .merge(events_per_session.rename("n_trials_events").reset_index(), on="USID", how="left")
    .fillna({"n_trials_parser": 0, "n_trials_events": 0})
    .astype({"n_trials_parser": int, "n_trials_events": int})
    .assign(parser_matches_events=lambda d: d["n_trials_parser"] == d["n_trials_events"])
)

print(f"parser matches raw Poke-event count in {trial_count_table['parser_matches_events'].sum()} / {len(trial_count_table)} sessions")
display_table(trial_count_table)

## 5. Derived Columns and Optional Filtering

By default all valid copied data are included. Add filters in `analysis_mask` when you want a subset, for example a single animal, day range, or cue condition.

In [ ]:
def extract_day_number(day_label):
    match = re.search(r"\d+", str(day_label))
    return int(match.group()) if match else np.nan


def to_bool_series(series):
    if series.dtype == object:
        return series.map(
            lambda value: False
            if pd.isna(value)
            else str(value).strip().lower() in {"true", "1", "yes"}
        )
    return series.fillna(False).astype(bool)


master_trials["day_num"] = master_trials["day"].map(extract_day_number)
master_trials["chosen_port"] = pd.to_numeric(master_trials["chosen_port"], errors="coerce")
master_trials["correct_port"] = pd.to_numeric(master_trials["correct_port"], errors="coerce")
master_trials["success"] = master_trials["success"].astype(bool)

half_n = N_PORTS // 2
signed_port_offset = ((master_trials["chosen_port"] - master_trials["correct_port"] + half_n) % N_PORTS) - half_n
master_trials["signed_error_angle"] = np.where(
    master_trials["chosen_port"] >= 0,
    signed_port_offset * (360.0 / N_PORTS),
    np.nan,
)
master_trials["absolute_error_angle"] = master_trials["signed_error_angle"].abs()

visual_draw_cols = [
    col for col in [
        "bonsai_landmark.draw",
        "bonsai_landmarkProximal.draw",
        "bonsai_grating.draw",
        "bonsai_firefly.draw",
    ]
    if col in master_trials.columns
]
master_trials["lights_on"] = False
if visual_draw_cols:
    master_trials["lights_on"] = pd.concat(
        [to_bool_series(master_trials[col]) for col in visual_draw_cols],
        axis=1,
    ).any(axis=1)

# Edit this mask for custom figure subsets.
analysis_mask = pd.Series(True, index=master_trials.index)
# analysis_mask &= master_trials["mouse"].isin(["FbR_M01569522"])
# analysis_mask &= master_trials["day_num"].between(1, 7)
# analysis_mask &= master_trials["lights_on"] == False

analysis_trials = master_trials.loc[analysis_mask].copy()
if analysis_trials.empty:
    raise ValueError("analysis_mask removed every trial; relax the filter before plotting.")

print(f"trials included  : {len(analysis_trials):,} / {len(master_trials):,}")
print(f"sessions included: {analysis_trials['USID'].nunique():,} / {master_trials['USID'].nunique():,}")
print(f"animals included : {', '.join(sorted(analysis_trials['mouse'].unique()))}")

display_table(
    analysis_trials[[
        "UTID", "mouse", "day", "day_num", "session", "trial_index",
        "outcome", "success", "chosen_port", "correct_port",
        "signed_error_angle", "absolute_error_angle",
        "outbound_duration", "inbound_duration", "lights_on",
    ]].head(40)
)

## 6. Session-Level Metrics

Each plotted point is a session. Connected markers show the per-animal mean for each day.

In [ ]:
session_keys = ["mouse", "day", "day_num", "USID"]

success_per_session = (
    analysis_trials
    .groupby(session_keys, as_index=False)
    .agg(success_rate=("success", "mean"), n_trials=("success", "size"))
)

rates_per_session = (
    analysis_trials
    .groupby(session_keys, as_index=False)
    .agg(
        miss_rate=("outcome", lambda s: (s == "miss").mean()),
        error_rate=("outcome", lambda s: (s == "fail").mean()),
        n_trials=("outcome", "size"),
    )
)

angle_per_session = (
    analysis_trials[analysis_trials["chosen_port"] >= 0]
    .groupby(session_keys, as_index=False)
    .agg(
        signed_error_angle=("signed_error_angle", "mean"),
        absolute_error_angle=("absolute_error_angle", "mean"),
        n_nonmiss_trials=("signed_error_angle", "size"),
    )
)

premature_per_session = count_premature_pokes_per_directory(
    base_dir=base_dir,
    sessions_df=sessions_df,
    start_marker=TRIAL_START_FOR_PARSER,
    poke_marker=POKE_MARKER,
    poke_extractor=parse_poke_outcome,
)
premature_per_session["day_num"] = premature_per_session["day"].map(extract_day_number)
premature_per_session = premature_per_session[
    premature_per_session["USID"].isin(analysis_trials["USID"].unique())
].copy()

observed_trials_per_session = (
    analysis_trials.groupby("USID").size().rename("n_observed_trials").reset_index()
)
premature_per_session = premature_per_session.merge(
    observed_trials_per_session,
    on="USID",
    how="left",
).fillna({"n_observed_trials": 0})

with np.errstate(divide="ignore", invalid="ignore"):
    denom = premature_per_session["n_observed_trials"].replace(0, np.nan)
    premature_per_session["premature_correct_rate"] = premature_per_session["n_premature_correct"] / denom
    premature_per_session["premature_incorrect_rate"] = premature_per_session["n_premature_incorrect"] / denom
    premature_per_session["premature_total_rate"] = premature_per_session["n_premature_total"] / denom

print("metric tables ready")
display_table(success_per_session.head(20))

## 7. Plot Helpers

In [ ]:
def mouse_colours(mice):
    cmap = plt.get_cmap("tab10")
    return {mouse: cmap(i % 10) for i, mouse in enumerate(sorted(mice))}


def metric_jitter(index, n_metrics, width=0.12):
    if n_metrics <= 1:
        return 0.0
    return (index - (n_metrics - 1) / 2) * width


def plot_session_metrics(ax, data, metrics, *, title, ylabel, ylim=None):
    """Plot session dots plus per-day animal means for one or more metrics."""
    if data.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)
        return

    colours = mouse_colours(data["mouse"].dropna().unique())
    metrics = list(metrics)

    for metric_index, metric in enumerate(metrics):
        column = metric["column"]
        label = metric.get("label", column)
        marker = metric.get("marker", "o")
        linestyle = metric.get("linestyle", "-")
        x_offset = metric_jitter(metric_index, len(metrics))

        for mouse, sub in data.groupby("mouse", sort=True):
            sub = sub.dropna(subset=["day_num", column])
            if sub.empty:
                continue

            colour = colours[mouse]
            scatter_kwargs = {
                "x": sub["day_num"] + x_offset,
                "y": sub[column],
                "color": colour,
                "marker": marker,
                "s": 28,
                "alpha": 0.30,
            }
            if marker not in {"x", "+", "1", "2", "3", "4", "|", "_"}:
                scatter_kwargs["edgecolor"] = "none"
            ax.scatter(**scatter_kwargs)

            per_day = sub.groupby("day_num", as_index=False)[column].mean()
            ax.plot(
                per_day["day_num"] + x_offset,
                per_day[column],
                color=colour,
                marker=marker,
                linestyle=linestyle,
                linewidth=1.8,
                markersize=6,
                label=f"{mouse} - {label}",
            )

    ax.set_title(title)
    ax.set_xlabel("Day")
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8, loc="best")


def save_figure(fig, name):
    if SAVE_FIGURES:
        figure_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(figure_dir / f"{name}.png", bbox_inches="tight", dpi=300)

## 8. Figure 1: Success Rate

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_session_metrics(
    ax,
    success_per_session,
    metrics=[{"column": "success_rate", "label": "success", "marker": "o"}],
    title="Success rate per session",
    ylabel="success rate",
    ylim=(0, 1.05),
)
fig.tight_layout()
save_figure(fig, "01_success_rate_per_session")
plt.show()

## 9. Figure 2: Miss Rate and Wrong-Poke Error Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_session_metrics(
    ax,
    rates_per_session,
    metrics=[
        {"column": "miss_rate", "label": "miss", "marker": "o", "linestyle": "-"},
        {"column": "error_rate", "label": "wrong poke", "marker": "s", "linestyle": "--"},
    ],
    title="Miss and wrong-poke error rates per session",
    ylabel="rate",
    ylim=(0, 1.05),
)
fig.tight_layout()
save_figure(fig, "02_miss_and_error_rates")
plt.show()

## 10. Figure 3: Signed and Unsigned Error Angle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
plot_session_metrics(
    axes[0],
    angle_per_session,
    metrics=[{"column": "absolute_error_angle", "label": "absolute", "marker": "o"}],
    title="Mean absolute error angle",
    ylabel="degrees",
)
plot_session_metrics(
    axes[1],
    angle_per_session,
    metrics=[{"column": "signed_error_angle", "label": "signed", "marker": "o"}],
    title="Mean signed error angle",
    ylabel="degrees",
)
axes[1].axhline(0, color="0.4", linewidth=1, linestyle=":")
fig.tight_layout()
save_figure(fig, "03_error_angles")
plt.show()

## 11. Figure 4: Premature Poke Rate

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_session_metrics(
    ax,
    premature_per_session,
    metrics=[
        {"column": "premature_correct_rate", "label": "correct premature", "marker": "^", "linestyle": "-"},
        {"column": "premature_incorrect_rate", "label": "incorrect premature", "marker": "x", "linestyle": "--"},
    ],
    title="Premature poke rate per observed trial",
    ylabel="premature pokes / observed trials",
)
fig.tight_layout()
save_figure(fig, "04_premature_poke_rates")
plt.show()

display_table(premature_per_session)

## 12. Exportable Analysis Tables

These tables are left in memory for downstream inspection: `sessions_df`, `bonsai_trials_df`, `behavioural_trials`, `master_trials`, `analysis_trials`, `trial_count_table`, and the per-session metric tables.

In [ ]:
summary = {
    "animals": sorted(analysis_trials["mouse"].unique()),
    "sessions": int(analysis_trials["USID"].nunique()),
    "trials": int(len(analysis_trials)),
    "days": sorted(int(day) for day in analysis_trials["day_num"].dropna().unique()),
}
summary